# Semáforo de descarga de agua

**Reto Auditoría FEMSA - Analítico de descarga de agua** · TI2006B Gpo 303 · Equipo 1

Este cuaderno calcula el semáforo que elegimos después de comparar 6 métodos. El semáforo responde la pregunta 1 del reto: **¿qué plantas y meses conviene revisar primero?**

Cada planta-mes se califica en las 3 validaciones que pidió Auditoría FEMSA y toma el color más grave:

| Validación | Verde | Amarillo | Rojo |
|---|---|---|---|
| Conciliación con Reporting Factory (solo efluente) | ≤ 0.5% o ≤ 10 m³ de diferencia | > 0.5% y > 10 m³ | > 10% y > 100 m³ |
| Continuidad: días sin lectura | 0 | 1 a 3 | 4 o más |
| Lecturas atípicas: % de días con cero, pico o valor constante | ≤ 5% | 5% a 10% | > 10% |

Además, si Reporting Factory capturó el mes en m³ en lugar de litros (regla R10), el mes queda como mínimo en amarillo.

Un rojo significa "revisar primero", no incumplimiento. Los umbrales son una propuesta y los validaremos con Auditoría FEMSA.

**Entradas:** `ConciliacionMensual.csv` y `RegistrosDiarios.csv`, exportados de `fact_conciliacion_mensual` y `fact_registros_diarios`.
**Salidas:** `semaforo_planta_mes.csv`, `matriz_prioridad_planta.csv` y las 5 tablas del modelo para Power BI.

## 1. Configuración

In [ ]:
import pandas as pd # tablas
import numpy as np # cálculos numéricos

# Umbrales del semáforo (se cambian aquí si Auditoría pide ajustarlos)
CONC_VERDE_PCT, CONC_ROJO_PCT = 0.5, 10 # % de diferencia con Reporting Factory
CONC_AMARILLO_M3, CONC_ROJO_M3 = 10, 100 # materialidad: m³ mínimos de diferencia
CONT_AMARILLO_MAX = 3 # días sin lectura: 1 a 3 amarillo, 4 o más rojo
ATIP_VERDE_PCT, ATIP_ROJO_PCT = 5, 10 # % de días con lecturas atípicas

# Pesos del puntaje de prioridad (suman 1)
PESO_CONC, PESO_ATIP, PESO_CONT = 0.50, 0.25, 0.25

cm = pd.read_csv("ConciliacionMensual.csv", parse_dates=["fecha_id"])
cm = cm.loc[:, ~cm.columns.duplicated()] # el export trae nombre_mes repetido
print("Planta-mes:", len(cm), "| Plantas:", cm["nombre_planta"].nunique(), "| Periodo:", cm["fecha_id"].min().date(), "a", cm["fecha_id"].max().date())

Planta-mes: 230 | Plantas: 23 | Periodo: 2026-01-01 a 2026-10-01


## 2. Calificación por validación

`0 = verde`, `1 = amarillo`, `2 = rojo`. La diferencia viene en litros, así que se divide entre 1,000 para pasarla a m³.

In [ ]:
pct = cm["pct_discrepancia"].abs()
dif_m3 = cm["diferencia_l"].abs() / 1000

# Conciliación con materialidad: la diferencia tiene que ser grande en % y en m³
cm["nivel_conciliacion"] = np.select(
    [(pct > CONC_ROJO_PCT) & (dif_m3 > CONC_ROJO_M3),
     (pct > CONC_VERDE_PCT) & (dif_m3 > CONC_AMARILLO_M3)],
    [2, 1], default=0)

# Continuidad: días del mes sin lectura del medidor
cm["nivel_continuidad"] = np.select(
    [cm["dias_faltantes"] == 0, cm["dias_faltantes"] <= CONT_AMARILLO_MAX],
    [0, 1], default=2)

# Lecturas atípicas: % de días en cero, con pico o con valor constante
cm["nivel_atipicas"] = np.select(
    [cm["pct_dias_anomalos"] <= ATIP_VERDE_PCT, cm["pct_dias_anomalos"] <= ATIP_ROJO_PCT],
    [0, 1], default=2)

# Error de unidad en Reporting Factory (R10): amarillo como mínimo
cm["nivel_unidad"] = cm["flag_posible_error_unidad"].astype(int)

cm[["nivel_conciliacion", "nivel_continuidad", "nivel_atipicas", "nivel_unidad"]].apply(pd.Series.value_counts).fillna(0).astype(int)

,nivel_conciliacion,nivel_continuidad,nivel_atipicas,nivel_unidad
0,140,212,199,224
1,67,6,19,6
2,23,12,12,0


## 3. Color final y puntaje de prioridad

In [ ]:
COLORES = {0: "Verde", 1: "Amarillo", 2: "Rojo"}

cm["nivel_semaforo"] = cm[["nivel_conciliacion", "nivel_continuidad", "nivel_atipicas", "nivel_unidad"]].max(axis=1)
cm["semaforo"] = cm["nivel_semaforo"].map(COLORES)

# Puntaje de 0 a 100 para ordenar dentro de cada color
cm["puntaje_prioridad"] = (100 * (
    PESO_CONC * np.minimum(pct / 20, 1) +
    PESO_ATIP * np.minimum(cm["pct_dias_anomalos"] / 20, 1) +
    PESO_CONT * np.minimum(cm["dias_faltantes"] / 5, 1))).round(1)

# Reporting Factory solo cuadra si se le suma el riego (FEMSA confirmó que no debe incluirlo)
cm["rf_incluye_riego"] = (pct > 0.5) & (cm["pct_discrepancia_con_riego"].abs() <= 0.5)

cm["semaforo"].value_counts()

,count
semaforo,
Verde,117
Amarillo,75
Rojo,38


## 4. Motivo de cada alerta

In [ ]:
def motivo(r):
    m = []
    if r["nivel_conciliacion"] > 0:
        m.append(f"diferencia de {abs(r['pct_discrepancia']):.1f}% ({abs(r['diferencia_l']) / 1000:,.0f} m³) con Reporting Factory")
    if r["rf_incluye_riego"]:
        m.append("Reporting Factory incluye riego")
    if r["nivel_unidad"]:
        m.append("Reporting Factory capturó en m³")
    if r["nivel_continuidad"] > 0:
        m.append(f"{int(r['dias_faltantes'])} días sin lectura")
    if r["nivel_atipicas"] > 0:
        m.append(f"{r['pct_dias_anomalos']:.0f}% de días atípicos")
    return "; ".join(m) if m else "sin observaciones"

cm["motivo"] = cm.apply(motivo, axis=1)

semaforo = (cm[["fecha_id", "planta_id", "nombre_planta", "periodo", "semaforo", "nivel_semaforo",
                "puntaje_prioridad", "motivo", "pct_discrepancia", "diferencia_l", "dias_faltantes",
                "pct_dias_anomalos", "rf_incluye_riego", "flag_posible_error_unidad"]]
            .sort_values(["nivel_semaforo", "puntaje_prioridad"], ascending=False))
semaforo.head(15)

,fecha_id,planta_id,nombre_planta,periodo,semaforo,nivel_semaforo,puntaje_prioridad,motivo,pct_discrepancia,diferencia_l,dias_faltantes,pct_dias_anomalos,rf_incluye_riego,flag_posible_error_unidad
223,2026-04-01,23,Veracruz,Veracruz-2026-04,Rojo,2,82.6,diferencia de 17.1% (59 m³) con Reporting Fact...,17.05,-59000.0,5,12.00,False,0
194,2026-05-01,20,Tapachula,Tapachula-2026-05,Rojo,2,70.2,diferencia de 23.7% (18 m³) con Reporting Fact...,23.68,18000.0,0,16.13,False,0
39,2026-10-01,4,Cd. Victoria,Cd. Victoria-2026-10,Rojo,2,69.0,diferencia de 17.6% (64 m³) con Reporting Fact...,17.58,-64000.0,5,0.00,False,0
193,2026-04-01,20,Tapachula,Tapachula-2026-04,Rojo,2,67.4,diferencia de 17.0% (18 m³) con Reporting Fact...,16.98,-18000.0,5,0.00,False,0
32,2026-03-01,4,Cd. Victoria,Cd. Victoria-2026-03,Rojo,2,66.0,diferencia de 16.4% (69 m³) con Reporting Fact...,16.39,-69000.0,5,0.00,False,0
76,2026-07-01,8,Cuernavaca,Cuernavaca-2026-07,Rojo,2,64.6,diferencia de 17.9% (745 m³) con Reporting Fac...,17.86,-745000.0,4,0.00,False,0
140,2026-01-01,15,Querétaro,Querétaro-2026-01,Rojo,2,63.4,diferencia de 17.4% (590 m³) con Reporting Fac...,17.38,-590000.0,4,0.00,False,0
57,2026-08-01,6,Coatepec,Coatepec-2026-08,Rojo,2,60.8,"diferencia de 14.3% (1,647 m³) con Reporting F...",14.34,-1647000.0,5,0.00,False,0
129,2026-10-01,13,Pacífico,Pacífico-2026-10,Rojo,2,58.0,diferencia de 13.2% (496 m³) con Reporting Fac...,13.22,-496000.0,5,0.00,False,0
97,2026-08-01,10,León,León-2026-08,Rojo,2,56.1,"diferencia de 14.4% (2,905 m³) con Reporting F...",14.45,-2905000.0,4,0.00,False,0


## 5. Matriz de prioridad por planta

In [ ]:
matriz = (cm.groupby("nombre_planta")
            .agg(meses_rojo=("nivel_semaforo", lambda s: (s == 2).sum()),
                 meses_amarillo=("nivel_semaforo", lambda s: (s == 1).sum()),
                 meses_verde=("nivel_semaforo", lambda s: (s == 0).sum()),
                 puntaje_promedio=("puntaje_prioridad", "mean"),
                 puntaje_max=("puntaje_prioridad", "max"))
            .round(1)
            .sort_values(["meses_rojo", "puntaje_promedio"], ascending=False))
matriz

,meses_rojo,meses_amarillo,meses_verde,puntaje_promedio,puntaje_max
nombre_planta,,,,,
Celaya,8,1,1,39.0,54.2
Tapachula,6,1,3,26.0,70.2
Pacífico,4,1,5,21.6,58.0
Querétaro,3,6,1,22.2,63.4
Coatepec,3,2,5,18.8,60.8
PozaRica,3,3,4,18.2,40.6
Cuernavaca,2,7,1,20.4,64.6
Cd. Victoria,2,1,7,18.8,69.0
Cuautitlán,2,1,7,13.7,50.0


In [ ]:
# Vista planta x mes: V = verde, A = amarillo, R = rojo
vista = cm.pivot_table(index="nombre_planta", columns=cm["fecha_id"].dt.month,
                       values="semaforo", aggfunc="first").loc[matriz.index]
vista.apply(lambda col: col.str[0])

fecha_id,1,2,3,4,5,6,7,8,9,10
nombre_planta,,,,,,,,,,
Celaya,A,V,R,R,R,R,R,R,R,R
Tapachula,V,V,V,R,R,R,R,R,A,R
Pacífico,V,R,R,V,V,V,V,A,R,R
Querétaro,R,A,A,A,A,A,A,R,R,V
Coatepec,V,V,V,V,A,A,V,R,R,R
PozaRica,A,A,R,V,V,R,R,V,V,A
Cuernavaca,A,A,A,A,V,A,R,A,R,A
Cd. Victoria,V,V,R,A,V,V,V,V,V,R
Cuautitlán,V,V,V,V,V,A,R,V,V,R


## 6. Comprobaciones

In [ ]:
# Resultados esperados con los datos de enero a octubre de 2026
assert len(cm) == 230
assert cm["semaforo"].value_counts().to_dict() == {"Verde": 117, "Amarillo": 75, "Rojo": 38}
assert cm["rf_incluye_riego"].sum() == 52
assert semaforo.iloc[0]["periodo"] == "Veracruz-2026-04" and semaforo.iloc[0]["puntaje_prioridad"] == 82.6
# Todo mes con error de unidad queda al menos en amarillo
assert (cm.loc[cm["nivel_unidad"] == 1, "nivel_semaforo"] >= 1).all()
print("Comprobaciones superadas.")

Comprobaciones superadas.


## 7. Exportar para Power BI

In [ ]:
semaforo.to_csv("semaforo_planta_mes.csv", index=False)
matriz.to_csv("matriz_prioridad_planta.csv")
print("Archivos generados: semaforo_planta_mes.csv y matriz_prioridad_planta.csv")

Archivos generados: semaforo_planta_mes.csv y matriz_prioridad_planta.csv


## 8. Preparar las tablas para Power BI

Power BI funciona mejor con un modelo dimensional: tablas de hechos con los números y tablas de dimensiones con los catálogos. Esta sección toma el semáforo que acabamos de calcular y las 2 tablas que exportamos de la base (`ConciliacionMensual.csv` y `RegistrosDiarios.csv`) y genera 5 archivos:

| Archivo | Qué tiene |
|---|---|
| `fact_semaforo_mensual.csv` | Una fila por planta-mes: volúmenes en m³, días, color, puntaje, causa principal y motivo |
| `fact_lecturas_diarias.csv` | Una fila por planta-día: descarga en m³ y tipo de día (normal, cero, pico, constante o sin lectura) |
| `dim_planta.csv` | Catálogo de plantas: nombre, si usa riego, tamaño y lugar en el ranking de prioridad |
| `dim_mes.csv` | Catálogo de meses |
| `dim_semaforo.csv` | Los 3 colores y qué significa cada uno |

No se calcula nada nuevo: solo se pasan litros a m³, los meses a español y se separan las tablas.

In [ ]:
MESES = ["Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio", "Julio", "Agosto",
         "Septiembre", "Octubre", "Noviembre", "Diciembre"]
NOMBRES = {"PozaRica": "Poza Rica", "SanCristobal": "San Cristóbal", "SanJuanDelRio": "San Juan del Río"}
COLOR_HEX = {0: "#2E9E6B", 1: "#F2B233", 2: "#D9483B"}

# causa principal: la validación que puso el color final
def causa(r):
    if r["nivel_semaforo"] == 0:
        return "Sin observaciones"
    for col, nombre in [("nivel_conciliacion", "Conciliación"), ("nivel_continuidad", "Continuidad"),
                        ("nivel_atipicas", "Lecturas atípicas")]:
        if r[col] == r["nivel_semaforo"]:
            return nombre
    return "Error de unidad en RF"

cm["causa_principal"] = cm.apply(causa, axis=1)

fact_mensual = pd.DataFrame({
    "fecha_mes": cm["fecha_id"].dt.strftime("%Y-%m-%d"), "planta_id": cm["planta_id"], "periodo": cm["periodo"],
    "descarga_bitacora_m3": cm["descarga_bit_l"] / 1000, "reporting_factory_m3": cm["rf_l"] / 1000,
    "rf_original_m3": cm["rf_original_l"] / 1000, "diferencia_m3": cm["diferencia_l"] / 1000,
    "pct_discrepancia": cm["pct_discrepancia"], "riego_bitacora_m3": cm["riego_bit_l"] / 1000,
    "pct_discrepancia_con_riego": cm["pct_discrepancia_con_riego"], "dias_mes": cm["dias_mes"],
    "dias_con_registro": cm["dias_con_registro"], "dias_sin_lectura": cm["dias_faltantes"],
    "dias_cero": cm["dias_cero"], "dias_pico": cm["dias_pico"], "dias_constante": cm["dias_constante"],
    "dias_anomalos": cm["dias_anomalos"], "pct_dias_anomalos": cm["pct_dias_anomalos"],
    "pct_continuidad": cm["pct_continuidad"], "rf_incluye_riego": cm["rf_incluye_riego"].astype(int),
    "error_unidad_rf": cm["nivel_unidad"], "nivel_conciliacion": cm["nivel_conciliacion"],
    "nivel_continuidad": cm["nivel_continuidad"], "nivel_atipicas": cm["nivel_atipicas"],
    "nivel_semaforo": cm["nivel_semaforo"], "semaforo": cm["semaforo"],
    "color_semaforo": cm["nivel_semaforo"].map(COLOR_HEX), "puntaje_prioridad": cm["puntaje_prioridad"],
    "causa_principal": cm["causa_principal"], "motivo": cm["motivo"]})

# dimensión planta: catálogo + ranking de la matriz de prioridad
dim_planta = (cm.groupby(["planta_id", "nombre_planta"])
                .agg(usa_riego=("usa_riego", "first"), vol=("descarga_bit_l", "mean")).reset_index())
dim_planta["planta"] = dim_planta["nombre_planta"].replace(NOMBRES)
dim_planta["usa_riego"] = np.where(dim_planta["usa_riego"], "Sí", "No")
dim_planta["descarga_mensual_promedio_m3"] = (dim_planta["vol"] / 1000).round(0)
dim_planta["tamano"] = pd.cut(dim_planta["descarga_mensual_promedio_m3"], [-1, 1000, 10000, 1e9],
                              labels=["Chica (<1,000 m³/mes)", "Mediana (1,000-10,000 m³/mes)", "Grande (>10,000 m³/mes)"])
dim_planta = dim_planta.merge(matriz.reset_index(), on="nombre_planta")
dim_planta = dim_planta.sort_values(["meses_rojo", "puntaje_promedio"], ascending=False)
dim_planta["ranking_prioridad"] = range(1, len(dim_planta) + 1)
dim_planta = dim_planta[["planta_id", "planta", "usa_riego", "tamano", "descarga_mensual_promedio_m3",
                         "ranking_prioridad", "meses_rojo", "meses_amarillo", "meses_verde",
                         "puntaje_promedio", "puntaje_max"]].sort_values("planta_id")

# dimensión mes, en español
dim_mes = pd.DataFrame({"fecha_mes": sorted(cm["fecha_id"].unique())})
dim_mes["anio"] = dim_mes["fecha_mes"].dt.year
dim_mes["mes_num"] = dim_mes["fecha_mes"].dt.month
dim_mes["mes"] = dim_mes["mes_num"].map(lambda i: MESES[i - 1])
dim_mes["mes_corto"] = dim_mes["mes"].str[:3]
dim_mes["trimestre"] = "T" + dim_mes["fecha_mes"].dt.quarter.astype(str)
dim_mes["mes_anio"] = dim_mes["mes_corto"] + " " + dim_mes["anio"].astype(str)
dim_mes["fecha_mes"] = dim_mes["fecha_mes"].dt.strftime("%Y-%m-%d")

# significado de cada color
dim_semaforo = pd.DataFrame({
    "nivel_semaforo": [0, 1, 2], "semaforo": ["Verde", "Amarillo", "Rojo"],
    "color_semaforo": [COLOR_HEX[0], COLOR_HEX[1], COLOR_HEX[2]],
    "significado": ["Sin observaciones relevantes",
                    "Revisar: diferencia moderada, huecos cortos o algunos días atípicos",
                    "Revisar primero: diferencia material, 4+ días sin lectura o >10% de días atípicos"]})

In [ ]:
# hechos diarios, a partir de RegistrosDiarios.csv (export de fact_registros_diarios)
rd = pd.read_csv("RegistrosDiarios.csv", parse_dates=["fecha_id"])

fact_diaria = pd.DataFrame({
    "fecha": rd["fecha_id"].dt.strftime("%Y-%m-%d"),
    "fecha_mes": rd["fecha_id"].dt.to_period("M").dt.to_timestamp().dt.strftime("%Y-%m-%d"),
    "planta_id": rd["planta_id"], "lectura_medidor_m3": rd["flujo_efluente"] / 1000,
    "descarga_diaria_m3": rd["diferencia_diaria_efluente"] / 1000,
    "descarga_promedio_dia_m3": rd["descarga_promedio_dia"] / 1000,
    "riego_diario_m3": rd["diferencia_diaria_riego"] / 1000, "limite_pico_m3": rd["limite_pico"] / 1000,
    "sin_lectura": rd["flag_sin_lectura"], "cero": rd["flag_cero"], "pico": rd["flag_pico"],
    "constante": rd["flag_constante"], "anomalia": rd["flag_anomalia"]})
fact_diaria["tipo_dia"] = np.select(
    [fact_diaria["sin_lectura"] == 1, fact_diaria["pico"] == 1, fact_diaria["constante"] == 1, fact_diaria["cero"] == 1],
    ["Sin lectura", "Pico", "Valor constante", "Cero"], default="Normal")

# comprobaciones: los totales no cambian al reorganizar
assert len(fact_mensual) == 230 and len(fact_diaria) == 6992
assert fact_mensual["nivel_semaforo"].value_counts().to_dict() == {0: 117, 1: 75, 2: 38}
assert fact_diaria["anomalia"].sum() == cm["dias_anomalos"].sum() == 179

for nombre, tabla in [("fact_semaforo_mensual", fact_mensual), ("fact_lecturas_diarias", fact_diaria),
                      ("dim_planta", dim_planta), ("dim_mes", dim_mes), ("dim_semaforo", dim_semaforo)]:
    tabla.to_csv(f"{nombre}.csv", index=False, encoding="utf-8-sig")
    print(f"{nombre}.csv: {tabla.shape[0]} filas")

fact_semaforo_mensual.csv: 230 filas
fact_lecturas_diarias.csv: 6992 filas
dim_planta.csv: 23 filas
dim_mes.csv: 10 filas
dim_semaforo.csv: 3 filas


## 9. Misma lógica en SQL (PostgreSQL)

Para dejar el semáforo dentro de la base, junto al modelo dimensional, se puede crear esta vista sobre `fact_conciliacion_mensual`:

```sql
CREATE OR REPLACE VIEW vw_semaforo_planta_mes AS
WITH niveles AS (
    SELECT f.*, p.nombre_planta,
        CASE WHEN ABS(pct_discrepancia) > 10 AND ABS(diferencia_L) / 1000 > 100 THEN 2
             WHEN ABS(pct_discrepancia) > 0.5 AND ABS(diferencia_L) / 1000 > 10 THEN 1
             ELSE 0 END AS nivel_conciliacion,
        CASE WHEN dias_faltantes = 0 THEN 0 WHEN dias_faltantes <= 3 THEN 1 ELSE 2 END AS nivel_continuidad,
        CASE WHEN pct_dias_anomalos <= 5 THEN 0 WHEN pct_dias_anomalos <= 10 THEN 1 ELSE 2 END AS nivel_atipicas,
        COALESCE(flag_posible_error_unidad, 0) AS nivel_unidad
    FROM fact_conciliacion_mensual f
    JOIN dim_planta p ON p.planta_id = f.planta_id
)
SELECT fecha_id, planta_id, nombre_planta, periodo,
       GREATEST(nivel_conciliacion, nivel_continuidad, nivel_atipicas, nivel_unidad) AS nivel_semaforo,
       CASE GREATEST(nivel_conciliacion, nivel_continuidad, nivel_atipicas, nivel_unidad)
            WHEN 0 THEN 'Verde' WHEN 1 THEN 'Amarillo' ELSE 'Rojo' END AS semaforo,
       ROUND(100 * (0.50 * LEAST(ABS(pct_discrepancia) / 20, 1)
                  + 0.25 * LEAST(pct_dias_anomalos / 20, 1)
                  + 0.25 * LEAST(dias_faltantes / 5.0, 1)), 1) AS puntaje_prioridad
FROM niveles;
```

## 10. Supuestos

- La conciliación usa solo el efluente, porque FEMSA confirmó que Reporting Factory no debe incluir el riego.
- La comparación contra el volumen autorizado queda fuera: el valor de referencia de la bitácora es un promedio de la planta y FEMSA hará esa validación manualmente.
- Umbrales, materialidad y pesos son propuesta del equipo, pendientes de validar con Auditoría FEMSA. Se cambian en la celda de configuración.